In [ ]:
"""
Billing Software for Om Green Field Associates
"""

import os
import datetime
from tkinter import *
from tkinter import ttk, messagebox
from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.lib.units import mm
from reportlab.pdfgen import canvas
from reportlab.platypus import Table, TableStyle
from num2words import num2words

# ------------------ Supplier Details ------------------
SUPPLIER = {
    "gst": "09BIPPK6152Bxxx",
    "name": "OM GREEN FIELD ASSOCIATES",
    "address": "KHOLAPHOOL, SIKANDRA, KANPUR",
    "pin": "208001",
    "state": "UTTAR PRADESH",
    "state_code": "09",
    "mobile": "8397070xxx, 9416369xxx",
    "bank_name": "State Bank of India",
    "account_no": "59180100503xxx",
    "ifsc": "UBIN05591xxx"
}

BILLS_FOLDER = "bills"
INVOICE_NO_FILE = "invoice_no.txt"


# ------------------ Helper functions ------------------
def load_invoice_no():
    if not os.path.exists(INVOICE_NO_FILE):
        with open(INVOICE_NO_FILE, "w") as f:
            f.write("100")  # starting invoice no (you can change)
        return 100
    with open(INVOICE_NO_FILE, "r") as f:
        try:
            return int(f.read().strip())
        except:
            return 100


def save_invoice_no(no):
    with open(INVOICE_NO_FILE, "w") as f:
        f.write(str(no))


def amount_to_words(amount):
    # num2words returns 'fourteen thousand one hundred seventy six'
    words = num2words(round(amount), to="cardinal", lang="en_IN")
    # Capitalize and add 'Rupees Only'
    return words.replace(" and ", " ").title() + " Rupees Only"


# ------------------ PDF Generation ------------------

def generate_pdf(invoice_no, invoice_date, supplier, customer, items, pdf_path):

    c = canvas.Canvas(pdf_path, pagesize=A4)
    width, height = A4

    LEFT = 20 * mm
    RIGHT = width - 20 * mm
    TOP = height - 20 * mm
    GAP = 5 * mm

    # ---------------- HEADER (TOP CENTER) ----------------
    c.setFont("Helvetica-Bold", 16)
    c.drawCentredString(width/2, TOP, "TAX INVOICE")

    # ---------------- SUPPLIER — TOP LEFT ----------------
    y = TOP - 10 * mm
    c.setFont("Helvetica-Bold", 11)
    c.drawString(LEFT, y, supplier["name"])

    c.setFont("Helvetica", 9)
    y -= GAP
    c.drawString(LEFT, y, f"GST No.: {supplier['gst']}")
    y -= GAP
    c.drawString(LEFT, y, f"Address: {supplier['address']}, PIN: {supplier['pin']}")
    y -= GAP
    c.drawString(LEFT, y, f"State: {supplier['state']} ({supplier['state_code']})")
    y -= GAP
    c.drawString(LEFT, y, f"Mobile: {supplier['mobile']}")

    # ---------------- RECEIVER — TOP RIGHT ----------------
    rx = RIGHT - 68*mm
    ry = TOP - 10 * mm

    c.setFont("Helvetica-Bold", 11)
    c.drawString(rx, ry, "Receiver Details:")

    c.setFont("Helvetica", 9)
    ry -= GAP
    c.drawString(rx, ry, f"Name: {customer.get('name','')}")
    ry -= GAP
    c.drawString(rx, ry, f"GSTN: {customer.get('gst','')}")
    ry -= GAP
    c.drawString(rx, ry, f"Address: {customer.get('address','')}")
    ry -= GAP
    c.drawString(rx, ry, f"Pin: {customer.get('pin','')}  State: {customer.get('state','')} ({customer.get('state_code','')})")

    # ---------------- INVOICE DETAILS ----------------
    ry -= GAP*2
    c.setFont("Helvetica-Bold", 10)
    c.drawString(rx, ry, f"Invoice No.: {invoice_no}")
    ry -= GAP
    c.drawString(rx, ry, f"Date: {invoice_date.strftime('%d-%m-%Y')}")

    # ---------------- TABLE ----------------
    table_y = ry - 10*mm

    COLS = [15*mm, 55*mm, 18*mm, 15*mm, 15*mm, 22*mm, 15*mm, 25*mm, 25*mm]

    data = [
        ["Sr", "Item", "HSN", "UOM", "Qty", "Rate", "Tax%", "Tax Amt", "Total"]
    ]

    for it in items:
        data.append([
            it["sr"],
            it["name"],
            it["hsn"],
            it["uom"],
            it["qty"],
            f"{it['rate']:.2f}",
            f"{it['tax_percent']:.2f}",
            f"{it['tax_amt']:.2f}",
            f"{it['amount']:.2f}"
        ])

    # totals row at end
    total_before = sum(it["qty"] * it["rate"] for it in items)
    total_tax = sum(it["tax_amt"] for it in items)
    total_after = total_before + total_tax

    data.append(["", "", "", "", "", "", "Total", f"{total_tax:.2f}", f"{total_after:.2f}"])

    table = Table(data, colWidths=COLS)
    table.setStyle(TableStyle([
        ("GRID", (0,0), (-1,-1), 0.5, colors.black),
        ("BACKGROUND", (0,0), (-1,0), colors.lightgrey),
        ("ALIGN", (0,0), (-1,-1), "CENTER"),
        ("VALIGN", (0,0), (-1,-1), "MIDDLE"),
        ("FONTSIZE", (0,0), (-1,-1), 8),
    ]))

    table_height = len(data) * 8 * mm
    table.wrapOn(c, width, height)
    table.drawOn(c, LEFT, table_y - table_height)

    y = table_y - table_height - 10*mm

    # ---------------- AMOUNT IN WORDS ----------------
    c.setFont("Helvetica-Bold", 10)
    words = amount_to_words(total_after)
    c.drawString(LEFT, y, f"Amount in Words: {words}")

    y -= 10*mm

    # ---------------- BANK DETAILS ----------------
    c.setFont("Helvetica-Bold", 10)
    c.drawString(LEFT, y, "Bank Details:")
    c.setFont("Helvetica", 9)
    y -= GAP
    c.drawString(LEFT, y, f"Bank: {supplier['bank_name']}")
    y -= GAP
    c.drawString(LEFT, y, f"Account No.: {supplier['account_no']}")
    y -= GAP
    c.drawString(LEFT, y, f"IFSC: {supplier['ifsc']}")

    # ---------------- SIGNATURE — BOTTOM LEFT ----------------
    c.setFont("Helvetica", 9)
    c.drawString(LEFT, 30*mm, "Authorised Signatory")

    c.save()


# ------------------ Tkinter GUI ------------------
class BillingApp:
    def __init__(self, root):
        self.root = root
        self.root.title("Billing Software - Om Green Field Associates")
        self.root.geometry("1000x700")
        self.items = []
        self.invoice_no = load_invoice_no()

        # Top frame - Customer details
        frame_top = LabelFrame(root, text="Customer Details", padx=8, pady=8)
        frame_top.pack(fill="x", padx=10, pady=6)

        Label(frame_top, text="Customer Name").grid(row=0, column=0, sticky=W)
        self.cust_name = Entry(frame_top, width=30)
        self.cust_name.grid(row=0, column=1, padx=4)

        Label(frame_top, text="Customer GSTN").grid(row=0, column=2, sticky=W)
        self.cust_gst = Entry(frame_top, width=20)
        self.cust_gst.grid(row=0, column=3, padx=4)

        Label(frame_top, text="Address").grid(row=1, column=0, sticky=W)
        self.cust_addr = Entry(frame_top, width=50)
        self.cust_addr.grid(row=1, column=1, columnspan=3, sticky=W, padx=4, pady=4)

        Label(frame_top, text="Pin Code").grid(row=2, column=0, sticky=W)
        self.cust_pin = Entry(frame_top, width=12)
        self.cust_pin.grid(row=2, column=1, sticky=W, padx=4)

        Label(frame_top, text="State").grid(row=2, column=2, sticky=W)
        self.cust_state = Entry(frame_top, width=20)
        self.cust_state.grid(row=2, column=3, sticky=W, padx=4)

        Label(frame_top, text="State Code").grid(row=2, column=4, sticky=W)
        self.cust_state_code = Entry(frame_top, width=6)
        self.cust_state_code.grid(row=2, column=5, sticky=W, padx=4)

        # Middle frame - Add item
        frame_mid = LabelFrame(root, text="Add Item", padx=8, pady=8)
        frame_mid.pack(fill="x", padx=10, pady=6)

        Label(frame_mid, text="Item Name").grid(row=0, column=0, sticky=W)
        self.item_name = Entry(frame_mid, width=40)
        self.item_name.grid(row=0, column=1, padx=4)

        Label(frame_mid, text="HSN Code").grid(row=0, column=2, sticky=W)
        self.item_hsn = Entry(frame_mid, width=10)
        self.item_hsn.grid(row=0, column=3, padx=4)

        Label(frame_mid, text="UOM").grid(row=1, column=0, sticky=W)
        self.item_uom = Entry(frame_mid, width=10)
        self.item_uom.grid(row=1, column=1, sticky=W)

        Label(frame_mid, text="Qty").grid(row=1, column=2, sticky=W)
        self.item_qty = Entry(frame_mid, width=8)
        self.item_qty.grid(row=1, column=3, sticky=W)

        Label(frame_mid, text="Unit Rate").grid(row=1, column=4, sticky=W)
        self.item_rate = Entry(frame_mid, width=12)
        self.item_rate.grid(row=1, column=5, sticky=W, padx=4)

        Label(frame_mid, text="Tax %").grid(row=1, column=6, sticky=W)
        self.item_tax = Entry(frame_mid, width=6)
        self.item_tax.grid(row=1, column=7, sticky=W, padx=4)
        self.item_tax.insert(0, "18")  # default

        Button(frame_mid, text="Add Item", command=self.add_item).grid(row=2, column=7, padx=6, pady=6)

        # Items table (Treeview)
        frame_table = LabelFrame(root, text="Items (Preview)", padx=8, pady=8)
        frame_table.pack(fill="both", expand=True, padx=10, pady=6)

        cols = ("sr", "name", "hsn", "uom", "qty", "rate", "tax%", "tax_amt", "amount")
        self.tree = ttk.Treeview(frame_table, columns=cols, show="headings", height=8)
        for c in cols:
            self.tree.heading(c, text=c.upper())
            self.tree.column(c, anchor=CENTER)
        self.tree.pack(fill="both", expand=True, padx=4, pady=4)

        Button(frame_table, text="Remove Selected", command=self.remove_selected).pack(pady=4)

        # Bottom frame - actions & totals
        frame_bottom = Frame(root)
        frame_bottom.pack(fill="x", padx=10, pady=8)

        # Totals display
        self.total_before_var = StringVar(value="0.00")
        self.total_tax_var = StringVar(value="0.00")
        self.total_after_var = StringVar(value="0.00")

        Label(frame_bottom, text="Total Before Tax:").grid(row=0, column=0, sticky=E)
        Label(frame_bottom, textvariable=self.total_before_var).grid(row=0, column=1, sticky=W)

        Label(frame_bottom, text="Total Tax:").grid(row=0, column=2, sticky=E)
        Label(frame_bottom, textvariable=self.total_tax_var).grid(row=0, column=3, sticky=W)

        Label(frame_bottom, text="Total After Tax:").grid(row=0, column=4, sticky=E)
        Label(frame_bottom, textvariable=self.total_after_var).grid(row=0, column=5, sticky=W)

        # Action buttons
        Button(frame_bottom, text="Generate PDF Invoice", command=self.generate_invoice, bg="#2196F3", fg="white").grid(row=1, column=0, pady=6)
        Button(frame_bottom, text="Clear All", command=self.clear_all, bg="#f44336", fg="white").grid(row=1, column=1, pady=6)
        Button(frame_bottom, text="Exit", command=root.destroy).grid(row=1, column=2, pady=6)

        # Initialize folder
        if not os.path.exists(BILLS_FOLDER):
            os.makedirs(BILLS_FOLDER)

        # load invoice no into label
        self.label_inv = Label(root, text=f"Next Invoice No: {self.invoice_no}", font=("Arial", 9))
        self.label_inv.pack(anchor="e", padx=12)

    def add_item(self):
        name = self.item_name.get().strip()
        hsn = self.item_hsn.get().strip()
        uom = self.item_uom.get().strip() or "nos"
        try:
            qty = float(self.item_qty.get())
            rate = float(self.item_rate.get())
            tax_percent = float(self.item_tax.get())
        except:
            messagebox.showerror("Error", "Qty, Rate and Tax must be numeric.")
            return

        if name == "":
            messagebox.showerror("Error", "Item name required.")
            return

        amount_before_tax = qty * rate
        tax_amt = amount_before_tax * (tax_percent / 100.0)
        total_amount = amount_before_tax + tax_amt

        sr = len(self.items) + 1
        item = {
            "sr": sr,
            "name": name,
            "hsn": hsn,
            "uom": uom,
            "qty": qty,
            "rate": rate,
            "tax_percent": tax_percent,
            "tax_amt": round(tax_amt, 2),
            "amount": round(total_amount, 2)
        }
        self.items.append(item)
        self.tree.insert("", "end", values=(item['sr'], item['name'], item['hsn'], item['uom'], item['qty'], f"{item['rate']:.2f}", f"{item['tax_percent']:.2f}", f"{item['tax_amt']:.2f}", f"{item['amount']:.2f}"))
        self.update_totals()

        # clear inputs
        self.item_name.delete(0, END)
        self.item_hsn.delete(0, END)
        self.item_uom.delete(0, END)
        self.item_qty.delete(0, END)
        self.item_rate.delete(0, END)
        self.item_tax.delete(0, END)
        self.item_tax.insert(0, "18")

    def remove_selected(self):
        sel = self.tree.selection()
        if not sel:
            return
        for s in sel:
            vals = self.tree.item(s, "values")
            sr = int(vals[0])
            # remove from items list
            self.items = [it for it in self.items if it['sr'] != sr]
            self.tree.delete(s)
        # reindex sr numbers in tree and items
        for idx, it in enumerate(self.items, start=1):
            it['sr'] = idx
        for i in self.tree.get_children():
            self.tree.delete(i)
        for it in self.items:
            self.tree.insert("", "end", values=(it['sr'], it['name'], it['hsn'], it['uom'], it['qty'], f"{it['rate']:.2f}", f"{it['tax_percent']:.2f}", f"{it['tax_amt']:.2f}", f"{it['amount']:.2f}"))

        self.update_totals()

    def update_totals(self):
        total_before = sum(it['qty'] * it['rate'] for it in self.items)
        total_tax = sum(it['tax_amt'] for it in self.items)
        total_after = total_before + total_tax

        self.total_before_var.set(f"₹{total_before:.2f}")
        self.total_tax_var.set(f"₹{total_tax:.2f}")
        self.total_after_var.set(f"₹{total_after:.2f}")

    def clear_all(self):
        self.items = []
        for i in self.tree.get_children():
            self.tree.delete(i)
        self.cust_name.delete(0, END)
        self.cust_gst.delete(0, END)
        self.cust_addr.delete(0, END)
        self.cust_pin.delete(0, END)
        self.cust_state.delete(0, END)
        self.cust_state_code.delete(0, END)
        self.update_totals()

    def generate_invoice(self):
        if not self.items:
            messagebox.showerror("Error", "Add at least one item.")
            return
        # gather customer info
        customer = {
            "name": self.cust_name.get().strip(),
            "gst": self.cust_gst.get().strip(),
            "address": self.cust_addr.get().strip(),
            "pin": self.cust_pin.get().strip(),
            "state": self.cust_state.get().strip(),
            "state_code": self.cust_state_code.get().strip()
        }
        if customer['name'] == "":
            messagebox.showerror("Error", "Enter customer name.")
            return

        # increment invoice no and save
        invoice_no = self.invoice_no
        invoice_date = datetime.datetime.now()
        pdf_name = f"invoice_{invoice_no}.pdf"
        pdf_path = os.path.join(BILLS_FOLDER, pdf_name)

        # Generate PDF
        generate_pdf(invoice_no, invoice_date, SUPPLIER, customer, self.items, pdf_path)

        # notify & increment
        messagebox.showinfo("Success", f"Invoice saved to {pdf_path}")
        self.invoice_no += 1
        save_invoice_no(self.invoice_no)
        self.label_inv.config(text=f"Next Invoice No: {self.invoice_no}")

        # Optional: clear items after saving
        self.clear_all()


# Run the app
if __name__ == "__main__":
    root = Tk()
    app = BillingApp(root)
    root.mainloop()
